In [1]:
import os
import sys

# 设置缓存目录路径
cache_dir = r'C:\work\Anaconda\envs\pytorch\cache'

# 确保目录存在
os.makedirs(cache_dir, exist_ok=True)

# 设置所有相关环境变量 - 使用新的环境变量
os.environ['HF_HOME'] = cache_dir
os.environ['HF_HUB_CACHE'] = cache_dir  
os.environ['HUGGINGFACE_HUB_CACHE'] = cache_dir

print(f"设置的缓存目录: {cache_dir}")

# 现在导入transformers
from transformers import pipeline

# 新版本不再有 TRANSFORMERS_CACHE，可以从环境变量获取
actual_cache_dir = os.environ.get('HF_HUB_CACHE', '')
print(f"实际缓存路径: {actual_cache_dir}")

# 验证路径是否正确
if cache_dir == actual_cache_dir:
    print("✅ 缓存路径设置成功！")
else:
    print(f"❌ 缓存路径设置失败！期望: {cache_dir}, 实际: {actual_cache_dir}")

# 运行情感分析
try:
    classifier = pipeline("sentiment-analysis")
    result = classifier(
        [
            "I've been waiting for a HuggingFace course my whole life.",
            "I hate this so much!",
        ]
    )
    print(f"\n分析结果: {result}")
except Exception as e:
    print(f"运行pipeline时出错: {e}")

# 检查缓存目录是否创建了文件
print(f"\n缓存目录内容:")
if os.path.exists(cache_dir):
    items = os.listdir(cache_dir)
    if items:
        for item in items:
            item_path = os.path.join(cache_dir, item)
            if os.path.isdir(item_path):
                print(f"  📁 {item}")
            else:
                print(f"  📄 {item}")
    else:
        print("   (空目录)")
else:
    print("   (目录不存在)")

# 检查并修复缓存目录权限
import subprocess
try:
    subprocess.run(['icacls', cache_dir, '/grant', 'Everyone:F', '/T'], shell=True, capture_output=True)
    print("\n✅ 缓存目录权限设置完成")
except Exception as e:
    print(f"\n 权限设置失败: {e}")

设置的缓存目录: C:\work\Anaconda\envs\pytorch\cache


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


实际缓存路径: C:\work\Anaconda\envs\pytorch\cache
✅ 缓存路径设置成功！


Device set to use cuda:0



分析结果: [{'label': 'POSITIVE', 'score': 0.9598046541213989}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

缓存目录内容:
  📁 .locks
  📁 datasets
  📁 datasets--bigcode--the-stack
  📁 datasets--code_search_net
  📁 datasets--conll2003
  📁 datasets--glue
  📁 datasets--imdb
  📁 datasets--lewtun--github-issues
  📁 datasets--liwu--MNBVC
  📁 datasets--pleisto--wikipedia-cn-20230720-filtered
  📁 datasets--wikitext
  📁 evaluate
  📁 metrics
  📁 models--bert-base-cased
  📁 models--bert-base-uncased
  📁 models--camembert-base
  📁 models--dbmdz--bert-large-cased-finetuned-conll03-english
  📁 models--distilbert--distilbert-base-cased-distilled-squad
  📁 models--distilbert--distilbert-base-uncased-finetuned-sst-2-english
  📁 models--distilbert-base-cased-distilled-squad
  📁 models--distilbert-base-uncased
  📁 models--distilbert-base-uncased-finetuned-sst-2-english
  📁 models--gpt2
  📁 models--roberta-base
  📁 models--sentence-transformers--multi-qa-mpnet-base-dot-v1
  📁 models--t5-small
  📁 models--ue

In [4]:
# model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

@dataclass
class GPTConfig:
    block_size: int = 512
    vocab_size: int = 21128
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1
    bias: bool = True

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        
        if not self.flash:
            print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))
    
    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        
        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
                dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v
        
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        
        self.apply(self._init_weights)
        
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is {self.config.block_size}"
        
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        
        for block in self.transformer.h:
            x = block(x)
        
        x = self.transformer.ln_f(x)
        
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=None, repetition_penalty=1.15, eos_token_id=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            
            if repetition_penalty != 1.0:
                for token_id in set(idx[0].tolist()):
                    logits[0, token_id] /= repetition_penalty
            
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            
            if eos_token_id is not None and idx_next.item() == eos_token_id:
                break
            
            idx = torch.cat((idx, idx_next), dim=1)
        
        return idx

In [7]:
# train_chinese_gpt.py
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from model import GPT, GPTConfig
import os
from tqdm import tqdm
import math

class TextDataset(Dataset):
    def __init__(self, file_path, tokenizer, block_size):
        self.tokenizer = tokenizer
        self.block_size = block_size
        self.data = []
        
        print(f"加载数据: {file_path}")
        with open(file_path, 'r', encoding='utf-8') as f:
            texts = f.readlines()
        
        # 获取EOS token ID
        if tokenizer.sep_token_id:
            eos_id = tokenizer.sep_token_id
        elif tokenizer.cls_token_id:
            eos_id = tokenizer.cls_token_id
        else:
            eos_id = tokenizer.pad_token_id or 0
        
        # 编码所有文本
        for text in tqdm(texts, desc="编码数据"):
            text = text.strip()
            if text:
                tokens = tokenizer.encode(text, add_special_tokens=False)
                if len(tokens) > 0:
                    tokens.append(eos_id)
                    self.data.extend(tokens)
        
        print(f"总token数: {len(self.data):,}")
    
    def __len__(self):
        if len(self.data) <= self.block_size:
            return 1
        return (len(self.data) - 1) // self.block_size
    
    def __getitem__(self, idx):
        start = idx * self.block_size
        end = start + self.block_size + 1
        
        if end > len(self.data):
            # 如果不够，就用前面的填充
            chunk = self.data[start:] + self.data[:max(0, end - len(self.data))]
        else:
            chunk = self.data[start:end]
        
        # 确保长度足够
        if len(chunk) < self.block_size + 1:
            pad_needed = self.block_size + 1 - len(chunk)
            chunk = chunk + [self.tokenizer.pad_token_id or 0] * pad_needed
        
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        
        return x, y

def train():
    # 1. 配置
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")
    
    # 2. 加载tokenizer
    print("加载中文tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
    
    # 设置特殊token
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    
    print(f"词表大小: {len(tokenizer)}")
    
    # 3. 模型配置
    config = GPTConfig(
        block_size=512,
        vocab_size=len(tokenizer),  # 使用实际词表大小
        n_layer=12,
        n_head=12,
        n_embd=768,
        dropout=0.1,
        bias=True
    )
    
    # 4. 创建模型
    print("创建模型...")
    model = GPT(config)
    model.to(device)
    
    # 打印参数数量
    total_params = sum(p.numel() for p in model.parameters())
    print(f"总参数: {total_params:,} ({total_params/1e6:.2f}M)")
    
    # 5. 加载数据集
    print("\n加载训练集...")
    train_dataset = TextDataset('data/train.txt', tokenizer, config.block_size)
    print("加载验证集...")
    val_dataset = TextDataset('data/val.txt', tokenizer, config.block_size)
    
    batch_size = 4  # 减小batch size以适应内存（124M模型需要较大内存）
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    print(f"训练批次数: {len(train_loader)}")
    print(f"验证批次数: {len(val_loader)}")
    
    # 6. 优化器和调度器
    learning_rate = 3e-4
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
    
    num_epochs = 20
    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(total_steps * 0.1)
    
    def get_lr(step):
        if step < warmup_steps:
            return learning_rate * step / warmup_steps
        else:
            progress = (step - warmup_steps) / (total_steps - warmup_steps)
            return learning_rate * 0.5 * (1 + math.cos(math.pi * progress))
    
    # 7. 训练循环
    best_val_loss = float('inf')
    global_step = 0
    
    print(f"\n开始训练 ({num_epochs} epochs)...")
    print("="*60)
    
    for epoch in range(num_epochs):
        # 训练阶段
        model.train()
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        
        for batch_idx, (x, y) in enumerate(progress_bar):
            x, y = x.to(device), y.to(device)
            
            # 动态学习率
            lr = get_lr(global_step)
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr
            
            optimizer.zero_grad()
            logits, loss = model(x, y)
            loss.backward()
            
            # 梯度裁剪
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            optimizer.step()
            
            total_loss += loss.item()
            global_step += 1
            
            # 更新进度条
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'lr': f'{lr:.2e}'
            })
            
            # 每100步打印一次
            if batch_idx % 100 == 0 and batch_idx > 0:
                print(f"\n  Step {global_step}: loss={loss.item():.4f}, lr={lr:.2e}")
        
        avg_train_loss = total_loss / len(train_loader)
        
        # 验证阶段
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in tqdm(val_loader, desc='验证'):
                x, y = x.to(device), y.to(device)
                _, loss = model(x, y)
                val_loss += loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        
        print(f"\nEpoch {epoch+1}: 训练损失={avg_train_loss:.4f}, 验证损失={avg_val_loss:.4f}")
        
        # 保存最佳模型
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_chinese_gpt.pth')
            print(f"✅ 保存最佳模型 (loss: {best_val_loss:.4f})")
        
        # 每5个epoch生成样例
        if (epoch + 1) % 5 == 0:
            model.eval()
            prompts = ["今天天气", "我喜欢", "世界上最好"]
            
            for prompt in prompts:
                input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
                with torch.no_grad():
                    output_ids = model.generate(
                        input_ids, 
                        max_new_tokens=30, 
                        temperature=0.8, 
                        top_k=40, 
                        repetition_penalty=1.15,
                        eos_token_id=tokenizer.sep_token_id
                    )
                generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
                print(f"\n生成样例 ({prompt}): {generated_text}")
            print("-"*60)
    
    print("\n✅ 训练完成！")
    return model

if __name__ == "__main__":
    # 检查数据文件是否存在
    if not os.path.exists('data/train.txt'):
        print("错误: 找不到 data/train.txt")
        exit(1)
    
    train()

使用设备: cuda
加载中文tokenizer...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

c:\work\Anaconda\envs\pytorch\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\work\Anaconda\envs\pytorch\cache\models--bert-base-chinese. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...


  Step 8623: loss=0.4175, lr=2.27e-07


Epoch 20/20:  92%|█████████▏| 402/438 [01:09<00:06,  5.78it/s, loss=0.3879, lr=1.63e-08]


  Step 8723: loss=0.3993, lr=1.72e-08


验证: 100%|██████████| 52/52 [00:02<00:00, 19.09it/s]



Epoch 20: 训练损失=0.3983, 验证损失=5.4535

生成样例 (今天天气): 今 天 天 气 那 个 比 华 冲 他 们 的 眼 镜 ， 并 无 顾 自 己 早 就 知 道 这 么 做 是 常 规 队 的 。

生成样例 (我喜欢): 我 喜 欢 真 不 能 把 它 送 什 么 ！ 可 这 事 儿 ！

生成样例 (世界上最好): 世 界 上 最 好 是 的 。 罗 辑 点 点 头 ， 用 拐 杖 一 个 弹 药 ， 把 这 小 心 翼 地 伸 向 上 去
------------------------------------------------------------

✅ 训练完成！


测试

In [10]:
# simple_test.py
import torch
import math
from transformers import AutoTokenizer
from model import GPT, GPTConfig

def load_model(model_path='best_chinese_gpt.pth'):
    """加载模型"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    
    config = GPTConfig(
        block_size=512,
        vocab_size=len(tokenizer),
        n_layer=12,
        n_head=12,
        n_embd=768,
        dropout=0.1,
        bias=True
    )
    
    model = GPT(config)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    return model, tokenizer, device

def test_on_test_set():
    """在测试集上评估模型"""
    print("加载模型...")
    model, tokenizer, device = load_model()
    
    # 读取测试集
    print("加载测试集...")
    with open('data/test.txt', 'r', encoding='utf-8') as f:
        test_texts = [line.strip() for line in f if line.strip()]
    
    print(f"测试集样本数: {len(test_texts)}\n")
    
    total_loss = 0
    results = []
    
    for i, text in enumerate(test_texts[:20]):  # 测试前20条
        # 编码并截断到模型最大长度
        input_ids = tokenizer.encode(text, add_special_tokens=False)
        
        # 关键修复：截断到 block_size - 1 (留一个位置给预测)
        max_len = model.config.block_size - 1
        if len(input_ids) > max_len:
            input_ids = input_ids[:max_len]
            text = tokenizer.decode(input_ids)  # 更新显示文本
        
        # 转换为tensor
        input_ids = torch.tensor([input_ids], device=device)
        targets = input_ids.clone()
        
        # 计算loss
        with torch.no_grad():
            _, loss = model(input_ids, targets)
        
        perplexity = math.exp(loss.item())
        total_loss += loss.item()
        
        # 显示文本（截断过长显示）
        display_text = text[:50] + '...' if len(text) > 50 else text
        
        results.append({
            'text': display_text,
            'loss': loss.item(),
            'perplexity': perplexity,
            'length': len(input_ids[0])
        })
        
        print(f"{i+1}. {display_text}")
        print(f"   长度: {len(input_ids[0])} tokens, Loss: {loss.item():.4f}, 困惑度: {perplexity:.2f}\n")
    
    # 平均结果
    avg_loss = total_loss / len(results)
    avg_perplexity = math.exp(avg_loss)
    
    print("="*50)
    print(f"📊 测试集结果 (共{len(results)}条):")
    print(f"   平均 Loss: {avg_loss:.4f}")
    print(f"   平均困惑度: {avg_perplexity:.2f}")
    print("="*50)
    
    # 生成一个示例（使用短文本作为prompt）
    print("\n📝 生成示例:")
    # 找一条短文本作为prompt
    short_texts = [t for t in test_texts if len(t) < 50]
    if short_texts:
        prompt = short_texts[0][:30]
    else:
        prompt = "今天天气"
    
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        output = model.generate(
            input_ids, 
            max_new_tokens=200,
            temperature=0.8,
            top_k=40,
            repetition_penalty=1.15,
            eos_token_id=tokenizer.sep_token_id
        )
    
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"输入: {prompt}")
    print(f"生成: {generated}")

if __name__ == "__main__":
    test_on_test_set()

加载模型...


Token indices sequence length is longer than the specified maximum sequence length for this model (714 > 512). Running this sequence through the model will result in indexing errors


加载测试集...
测试集样本数: 1288

1. “我不能容忍这个白痴了！”上校站起来大叫。
   长度: 21 tokens, Loss: 6.8813, 困惑度: 973.88

2. “我当然更不怕，公主，我可以跟着你到海的尽头，到世界尽头。”
   长度: 30 tokens, Loss: 7.9846, 困惑度: 2935.47

3. “尽量近一些，带有行星，最好是类地行星。”云天明看着星图说。
   长度: 30 tokens, Loss: 7.0792, 困惑度: 1187.04

4. 外面有不安的喧哗声，侍卫报告发生了月食。这是再明白不过的凶兆，因为在千年的风雨中有这样一句格言：只要...
   长度: 120 tokens, Loss: 7.2839, 困惑度: 1456.62

5. “那乱纪元会持续多长时间呢？”
   长度: 15 tokens, Loss: 6.3367, 困惑度: 564.93

6. “呵呵，你想得很有意思，但愿如此……看，这就是存放文物的地方，一共有三个这样的大厅。”
   长度: 43 tokens, Loss: 6.6686, 困惑度: 787.32

7. 随着空气渐渐稀薄，彩虹消失了，云雾也越来越淡，空间变得越来越透明，小宇宙的太空显现出来，它与大宇宙的...
   长度: 251 tokens, Loss: 7.4537, 困惑度: 1726.29

8. 这声音来自程心身后二楼的一个阳台，她转身看到了站的阳台上那个男人，不是现在女性化的男性，而是过去真正...
   长度: 203 tokens, Loss: 7.2646, 困惑度: 1428.83

9. 苏醒的过程很长，程心的意识是一点一点渐渐恢复的，当她的记忆和视力恢复后，知道的第一件事就是神经元计算...
   长度: 101 tokens, Loss: 7.2812, 困惑度: 1452.69

10. 超新星纪元第4个小时
   长度: 10 tokens, Loss: 5.7995, 困惑度: 330.13

11. 叶文洁：您很聪明。
   长度: 9 tokens, Loss: 9.4610, 困惑度: 12848.46

12. “我们想要一个好玩儿的世界！”
   长度: 15 tokens, Loss: 5.8571, 困惑度

互动！！

In [12]:
import torch
from transformers import AutoTokenizer
from model import GPT, GPTConfig

def load_model(model_path='best_chinese_gpt.pth'):
    """加载训练好的模型"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")
    
    # 加载tokenizer
    tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    if tokenizer.sep_token is None:
        tokenizer.add_special_tokens({'sep_token': '[SEP]'})
    
    # 模型配置
    config = GPTConfig(
        block_size=512,
        vocab_size=len(tokenizer),
        n_layer=12,
        n_head=12,
        n_embd=768,
        dropout=0.1,
        bias=True
    )
    
    # 加载模型
    model = GPT(config)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    print(f"✅ 模型加载成功！\n")
    
    return model, tokenizer, device

def generate_text(model, tokenizer, device, prompt):
    """生成文本 - 使用固定默认参数"""
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    # 检查长度
    current_len = input_ids.shape[1]
    max_new_tokens = min(300, 512 - current_len)  # 默认生成300个token
    
    if max_new_tokens <= 0:
        return "提示词太长，请缩短后重试"
    
    with torch.no_grad():
        output = model.generate(
            input_ids, 
            max_new_tokens=max_new_tokens,
            temperature=0.8,           # 固定温度
            top_k=40,                  # 固定Top-K
            repetition_penalty=1.15,   # 固定重复惩罚
            eos_token_id=tokenizer.sep_token_id
        )
    
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    return generated

def main():
    print("\n" + "="*60)
    print("🤖 三体小说助手")
    print("="*60)
    
    # 加载模型
    model, tokenizer, device = load_model()
    
    print("📖 使用说明:")
    print("  - 直接输入中文提示词，按回车生成")
    print("  - 输入 'quit' 或 'exit' 退出")
    print("  - 输入 'help' 显示帮助")
    print("="*60 + "\n")
    
    while True:
        # 获取用户输入
        user_input = input("🎤 请输入提示词: ").strip()
        
        if not user_input:
            continue
        
        # 退出命令
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("👋 再见！")
            break
        
        # 帮助
        if user_input.lower() == 'help':
            print("\n💡 直接输入任何中文文本，模型会续写内容")
            print("   例如: '今天天气' 或 '我喜欢'")
            print("   输入 'quit' 退出程序\n")
            continue
        
        # 生成文本
        print("🤖 生成中...")
        
        try:
            generated = generate_text(model, tokenizer, device, user_input)
            print(f"\n✨ 生成结果:\n{generated}\n")
            print("-"*60)
        except Exception as e:
            print(f"❌ 生成出错: {e}\n")

if __name__ == "__main__":
    main()


🤖 三体小说助手
使用设备: cuda
✅ 模型加载成功！

📖 使用说明:
  - 直接输入中文提示词，按回车生成
  - 输入 'quit' 或 'exit' 退出
  - 输入 'help' 显示帮助

🤖 生成中...

✨ 生成结果:
逻 辑 的 命 令 不 可 背 叛 ， 但 是 没 人 会 感 激 他 这 东 西 确 认 同 样 的 。 罗 辑 说 了 一 眼 ， 史 强 从 车 中 拿 起 车 开 走 动 ， 机 构 筑 上 方 向 的 弹 箱 像 飞 行 员 可 能 也 是 空 旷 的 火 架 作 响 飞 机 群 中 ， 飞 机 械 臂 很 快 放 射 线 上 去 。 几 分 钟 后 ， 罗 辑 看 到 史 强 拉 着 步 枪 上 的 车 和 罗 辑 发 现 了 他 还 是 警 察 ， 当 两 人 还 要 是 从 烟 盒 中 掏 出 双 肩 膀 ， 又 转 动 着 罗 辑 听 得 很 慢 了 ， 然 后 把 餐 刀 刺 刀 地 扔 下 来 的 步 话 。 再 次 下 了 ， 他 们 用 手 抓 住 脚 步 站 在 空 气 带 回 忆 ， 那 个 腿 顶 上 飞 过 地 面 的 沙 声 地 看 上 去.... 坎 特 ， 并 把 枪 身 于 是 一 堆 炸 声 从 服 上 的 衣 衫 破 碎 片 ， 使 这 边 缘 的 小 片 黑 黄 色 ， 有 装 满 了 一 切 都 是 在 惊 恐 惧 地 掠 过 光 。 罗 辑 听 到 后 的 心 翼 地 摇 头 盔 内 大 史 以 前 ， 罗 辑 把 他 才 说 出 来 ， 罗 辑 在 罗 辑 第 一 次 见 了 自 己 一 个 人 。

------------------------------------------------------------
🤖 生成中...

✨ 生成结果:
你 会 是 三 体 人 派 来 的 间 谍 吗 ？ 没 人 知 道 他 们 还 有 过 去 。 叶 文 洁 说 话 ： 太 阳 ， 我 在 我 们 那 个 夜 空 ， 就 是 三 体 文 明 的 宇 宙 在 其 中 的 死 寂 静 静 地 ， 一 切 都 没 有 出 现 了 大 的 波 动 ， 只 能 看 到 他 们 的 声 音 传 递 出 来 ， 也 不 管 这 儿 就 是 死 后 才 能 想 出 两 个 质 子 